## Model Training

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

%pwd

In [ ]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

#### Import the CSV Data as Pandas DataFrame

In [ ]:
df = pd.read_excel('notebook/Dataset/Dataset Disssertation.xlsx')

#### Show Top 5 Records

In [ ]:
df.head()

#### Remove all Unsampled Rows

In [ ]:
df = df[df['Sample_taken'] == 'Sampled']
df.shape

#### Drop Unneeded Features

In [ ]:
df= df.drop(columns=['WP_ID','DataType','Date_Assessment_Original','SURVEY_DETAIL_ID','COUNTRY','Comment','HCO3','Corrected_HCO3','Sample_taken'],axis=1)
df.shape

In [ ]:
df = df.drop(columns=['Date_Assessment','Time_Assessment'])
df.sample(5)

#### Check Missing Values

In [ ]:
df.isna().sum()

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
columns_to_impute = ['Total Iron (mg/l)', 'Tryptophan_Probe_?gL']
df[columns_to_impute] = imputer.fit_transform(df[columns_to_impute])

In [ ]:
df.isna().sum()

#### Preparing X and Y variables

In [ ]:
X = df.drop(columns=['pH'],axis=1)

In [ ]:
X.head()

In [ ]:
y = df['pH']

In [ ]:
y

#### Creating Pipeline with Column Transformer

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Create Column Transformer with 3 types of transformers
num_features = X.select_dtypes(exclude="object").columns
# cat_features = X.select_dtypes(include="object").columns

numeric_transformer = StandardScaler()
# oh_transformer = OneHotEncoder()

# Responsible for tranforming the data columns
preprocessor = ColumnTransformer(
    [
        # ("OneHotEncoder", oh_transformer, cat_features),
         ("StandardScaler", numeric_transformer, num_features),        
    ]
)

In [ ]:
X = preprocessor.fit_transform(X)

In [ ]:
X.shape

In [ ]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

#### Create an Evaluate Function to give all metrics after model Training

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

### Results

In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

#### The models are not performing well, for this we perform outlier detection and handling techniques.

##### Box_Plot for Outliers Detection

In [ ]:
# Box_Plot for Outliers Detection

sns.set_style("whitegrid")
plt.figure(figsize=(8, 6))

# Create a boxplot for each column in df_knn_cont
sns.boxplot(data=df, palette="Set3", fliersize=5, linewidth=1.5)

# Add title and labels
plt.title('Boxplot of Data with Outliers', fontsize=16)
plt.xlabel('Columns', fontsize=14)
plt.ylabel('Values', fontsize=14)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
sns.despine()

plt.grid(True, linestyle='--', alpha=0.7)

# Show plot
plt.tight_layout()
# plt.savefig('boxplot1.png')
plt.show()

In [ ]:
# Copy the original DataFrame
df_knn_rest = df.copy()

# Calculate Quartiles and IQR for all columns
Q1 = df_knn_rest.quantile(0.25)
Q3 = df_knn_rest.quantile(0.75)
IQR = Q3 - Q1

# Compute lower and upper bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Replace outliers with median for each column
for column in df_knn_rest.columns:
    # Identify outliers
    outliers = (df_knn_rest[column] < lower_bound[column]) | (df_knn_rest[column] > upper_bound[column])
    
    # Replace outliers with median
    median_value = df_knn_rest[column].median()
    df_knn_rest.loc[outliers, column] = median_value

# Check if any column still contains outliers
outliers_after_replacement = (df_knn_rest < lower_bound) | (df_knn_rest > upper_bound)

if outliers_after_replacement.any().any():
    print("Some columns still contain outliers after replacement.")
    print("Columns with outliers:")
    print(outliers_after_replacement.any())
else:
    print("No columns contain outliers after replacement.")


In [ ]:
# Restore back to Dataframe
df_restored = pd.DataFrame(df_knn_rest, columns=df.columns)

# Print the restored DataFrame
print("Restored DataFrame:")
df_restored.head()

In [ ]:
# Box_Plot for Outliers Detection

sns.set_style("whitegrid")
plt.figure(figsize=(8, 6))

# Create a boxplot for each column in df_knn_cont
sns.boxplot(data=df_restored, palette="Set3", fliersize=5, linewidth=1.5)

# Add title and labels
plt.title('Boxplot of Data with Outliers', fontsize=16)
plt.xlabel('Columns', fontsize=14)
plt.ylabel('Values', fontsize=14)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
sns.despine()

plt.grid(True, linestyle='--', alpha=0.7)

# Show plot
plt.tight_layout()
# plt.savefig('boxplot1.png')
plt.show()

In [ ]:
X_new = df_restored.drop(columns=['pH'],axis=1)
y_new = df_restored['pH']

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Create Column Transformer with 3 types of transformers
num_features = X_new.select_dtypes(exclude="object").columns
# cat_features = X.select_dtypes(include="object").columns

numeric_transformer = StandardScaler()
# oh_transformer = OneHotEncoder()

# Responsible for tranforming the data columns
preprocessor = ColumnTransformer(
    [
        # ("OneHotEncoder", oh_transformer, cat_features),
         ("StandardScaler", numeric_transformer, num_features),        
    ]
)

In [ ]:
X_new = preprocessor.fit_transform(X_new)
X_new

In [ ]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_new,y_new,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

#### Results

In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

## CatBoost Regression

In [ ]:
lin_model = LinearRegression()
lin_model = lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test)
score = r2_score(y_test, y_pred)*100
print(" Accuracy of the model is %.2f" %score)

## Plot y_pred and y_test

In [ ]:
plt.scatter(y_test,y_pred)
plt.xlabel('Actual')
plt.ylabel('Predicted')

In [ ]:
sns.regplot(x=y_test,y=y_pred,ci=None,color ='red');

#### Difference between Actual and Predicted Values

In [ ]:
pred_df=pd.DataFrame({'Actual Value':y_test,'Predicted Value':y_pred,'Difference':y_test-y_pred})
pred_df